In [1]:
import pandas as pd
import numpy as np
import csv
from unidecode import unidecode
from datetime import datetime


ARQUIVO_GRID = "/lakehouse/default/Files/raw_cadastro_carta/grid_carta_servicos_santos.csv"
ARQUIVO_BD = "/lakehouse/default/Files/raw_cadastro_carta/bd_carta_servicos_santos.csv"


def carregar_tratar_bd():
    bd_carta = pd.read_csv(
        ARQUIVO_BD,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )

    bd_carta = (
        bd_carta
        .drop(columns=["DATA DE ATUALIZAÇÃO"])
        .rename(columns={"Data-Hora da inserção/atualização dos dados (mais recente)": "data_de_atualizacao"})
    )

    bd_carta.columns = (
        bd_carta.columns.str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(":", "")
        .str.replace("º", "")
    )
    bd_carta.columns = [unidecode(col) for col in bd_carta.columns]

    bd_carta["status_tramitacao"] = "Finalizado"
    bd_carta["servico"] = "Cadastro de carta de serviço"

    bd_carta = bd_carta.drop(columns="area_executora").rename(
        columns={
            "nome": "nome_do_servico",
            "secretaria_responsavel": "area_responsavel",
        }
    )
    bd_carta['data_de_atualizacao'] = pd.to_datetime(bd_carta['data_de_atualizacao'], dayfirst=True)
    return bd_carta


def carregar_tratar_grid():
    grid_carta = pd.read_csv(
        ARQUIVO_GRID,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )

    grid_carta = grid_carta.drop(
        columns=["Nome:", "Área executora:", "Unnamed: 14", "Descrição:"]
    )

    grid_carta.columns = (
        grid_carta.columns.str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(":", "")
        .str.replace("º", "")
    )
    grid_carta.columns = [unidecode(col) for col in grid_carta.columns]

    grid_carta_cleaned = grid_carta.query("status in ('Em atendimento', 'Pendente')").drop(
        columns=["executor_atual", "etapa_atual", "solicitante", "data_de_finalizacao"]
    ).rename(
        columns={
            "nome": "nome_do_servico",
            "secretaria_responsavel": "area_responsavel",
            "status": "status_tramitacao",
        }
    )
    grid_carta_cleaned["id_do_servico"] = ""

    return grid_carta, grid_carta_cleaned


def gerar_bd_final(bd_carta, grid_carta_cleaned):
    df_bi_cadastro_carta = pd.concat([bd_carta, grid_carta_cleaned], axis=0)

    # data_consolidada = se finalizado: data_finalizacao, se em aberto: data_solicitacao
    df_bi_cadastro_carta["data_consolidada"] = np.where(
        df_bi_cadastro_carta["data_de_atualizacao"].isna(),
        df_bi_cadastro_carta["data_solicitacao"],
        df_bi_cadastro_carta["data_de_atualizacao"],
    )

    for col in ["data_de_atualizacao", "data_solicitacao", "data_consolidada"]:
        df_bi_cadastro_carta[col] = pd.to_datetime(
            df_bi_cadastro_carta[col], dayfirst=True
        )

    df_bi_cadastro_carta["sigla_area_responsavel"] = df_bi_cadastro_carta[
        "area_responsavel"
    ].str.split(" -", expand=True)[0]

    df_bi_cadastro_carta["dias_desde_atualizacao"] = (
        pd.to_datetime(datetime.now()) - df_bi_cadastro_carta["data_consolidada"]
    ).dt.days

    def categorizar_dias_atualizacao(dias):
        if dias < 30:
            return "Menor que 30 dias"
        elif dias < 60:
            return "Entre 30 e 60 dias"
        elif dias < 90:
            return "Entre 60 e 90 dias"
        elif dias < 120:
            return "Entre 90 e 120 dias"
        elif dias < 365:
            return "Entre 120 e 365 dias"
        else:
            return "Maior que 365 dias"

    df_bi_cadastro_carta["periodo_atualizacao"] = df_bi_cadastro_carta[
        "dias_desde_atualizacao"
    ].apply(categorizar_dias_atualizacao)

    df_bi_cadastro_carta["id_do_servico"] = np.where(
        (df_bi_cadastro_carta["id_do_servico"].notna())
        & (df_bi_cadastro_carta["status_tramitacao"] != "Finalizado"),
        np.nan,
        df_bi_cadastro_carta["id_do_servico"],
    )

    df_bi_cadastro_carta = df_bi_cadastro_carta.query(
        "nome_do_servico.notna()"
    ).reset_index(drop=True)


    mes_dict = {
        1: "janeiro",
        2: "fevereiro",
        3: "março",
        4: "abril",
        5: "maio",
        6: "junho",
        7: "julho",
        8: "agosto",
        9: "setembro",
        10: "outubro",
        11: "novembro",
        12: "dezembro"
    }


    df_bi_cadastro_carta["mes_solicitacao"] = df_bi_cadastro_carta['data_solicitacao'].dt.month
    df_bi_cadastro_carta["ano_solicitacao"] = df_bi_cadastro_carta['data_solicitacao'].dt.year
    df_bi_cadastro_carta["mes_solicitacao_txt"] = df_bi_cadastro_carta["mes_solicitacao"].map(mes_dict)

    df_bi_cadastro_carta["mes_atualizacao"] = df_bi_cadastro_carta['data_de_atualizacao'].dt.month
    df_bi_cadastro_carta["ano_atualizacao"] = df_bi_cadastro_carta['data_de_atualizacao'].dt.year
    df_bi_cadastro_carta["mes_atualizacao_txt"] = df_bi_cadastro_carta["mes_atualizacao"].map(mes_dict)

    df_bi_cadastro_carta["mes_consolidado"] = df_bi_cadastro_carta['data_consolidada'].dt.month
    df_bi_cadastro_carta["ano_consolidado"] = df_bi_cadastro_carta['data_consolidada'].dt.year
    df_bi_cadastro_carta["mes_consolidado_txt"] = df_bi_cadastro_carta["mes_consolidado"].map(mes_dict)

    return df_bi_cadastro_carta


def gerar_grid_final(grid):

    grid = grid.rename(
        columns={"nome": "nome_do_servico", "secretaria_responsavel": "area_responsavel"}
    )

    grid['sigla_area_responsavel'] = grid['area_responsavel'].str.split(" -", expand=True)[0]

    grid['id_do_servico'] = np.nan

    grid['data_solicitacao'] = pd.to_datetime(grid['data_solicitacao'], dayfirst=True)

    grid = grid[
        [
            "n_da_solicitacao",
            "nome_do_servico",
            "data_solicitacao",
            "id_do_servico",
            "etapa_atual",
            "executor_atual",
            "solicitante",
            "categoria",
            "area_responsavel",
            "sigla_area_responsavel",
        ]
    ].drop_duplicates().sort_values("nome_do_servico")

    mes_dict = {
        1: "janeiro",
        2: "fevereiro",
        3: "março",
        4: "abril",
        5: "maio",
        6: "junho",
        7: "julho",
        8: "agosto",
        9: "setembro",
        10: "outubro",
        11: "novembro",
        12: "dezembro"
    }


    grid["mes_solicitacao"] = grid['data_solicitacao'].dt.month
    grid["ano_solicitacao"] = grid['data_solicitacao'].dt.year
    grid["mes_solicitacao_txt"] = grid["mes_solicitacao"].map(mes_dict)

    return grid


bd_carta = carregar_tratar_bd()
grid_carta, grid_carta_cleaned = carregar_tratar_grid()
df_bi_cadastro_carta = gerar_bd_final(bd_carta, grid_carta_cleaned)
grid = gerar_grid_final(grid_carta)

StatementMeta(, 7c5e0e00-e5cd-4b7e-a057-dfe9825b531a, 3, Finished, Available, Finished, False)

In [2]:
grid_carta.query("nome == 'Vistoria anual de caminhão de aluguel'")

StatementMeta(, 7c5e0e00-e5cd-4b7e-a057-dfe9825b531a, 4, Finished, Available, Finished, False)

,n_da_solicitacao,servico,secretaria_responsavel,categoria,nome,solicitante,data_solicitacao,data_de_finalizacao,etapa_atual,status,executor_atual
579,823594,CADASTRO DE CARTA DE SERVIÇO,CET - COMPANHIA DE ENGENHARIA DE TRÁFEGO,TRÂNSITO E MOBILIDADE URBANA,Vistoria anual de caminhão de aluguel,ROSANGELA CANDIDO SALGUEIRO,26/08/2025 11:23,26/08/2025 11:25,ENCERRAMENTO,Finalizado,ADMINISTRAACTO ADMINISTRAACTO


In [3]:
# escrita da tabela no lh
df_bi_cadastro_carta_sdf = spark.createDataFrame(df_bi_cadastro_carta)
(
    df_bi_cadastro_carta_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_carta_servicos")
)


grid_sdf = spark.createDataFrame(grid)
(
    grid_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_carta_servicos_atualizacoes")
)

StatementMeta(, 7c5e0e00-e5cd-4b7e-a057-dfe9825b531a, 5, Finished, Available, Finished, True)